# FF49 — Neural CDE Diffusion

Daily value-weighted Kenneth French **49 Industry Portfolios** (`ken_french_49`, 49 assets).
The retained Neural CDE history encoder uses RK4 with four fixed steps per interval; its final state conditions the Gaussian diffusion model. The diffusion target remains v-prediction with the original MSE objective.

Configuration: **`configs/cde_ff49.yaml`**, seed **42**, lookback **60**, horizon **1**, total **50 epochs**.
Requested sample: **1958-01-01–2023-12-31**. Training targets end **1999-12-31**;
validation targets are in **2000–2004**; **2005–2023** is held out and is never evaluated here.

**Primary high-dimensional forecasting experiment.** Complete-case FF49 coverage begins in July 1969, despite the requested 1958 start. The verified archive gives 7,648 training, 1,256 validation, and 4,781 test windows. This historical coverage differs from KF12.

Returns are arithmetic decimal returns minus aligned daily French RF. Dates with missing
portfolios are dropped without imputation. Windows are segmented at gaps exceeding **7 calendar days**;
smaller omitted dates may remain within a segment. Standardization is fitted on training data only.
The smoke report below gives the current archive's actual counts and date ranges.

Run cells interactively: setup → tests → restore → two-epoch pilot → inspect losses → resume full
training → validation evaluation. Select training and backup cells when ready; **Run All launches
the full experiment and upload**. Pilot and full training continue the same run in
`/kaggle/working/ff49_cde_clean`. Saved files include `config.yaml`, `standardizer.npz`, `last.pt`,
`best.pt`, `history.csv`, and (after completion) `summary.json`; evaluation adds validation scenarios
and probabilistic metrics. No results are assumed in advance.

## Setup

In Kaggle settings enable **Internet** and select a **GPU** accelerator. The synchronization
cell follows the KF12 notebooks and resets the Kaggle checkout to the selected remote branch.
Change only `BRANCH` to `main` after merging. The branch and commit are printed below.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/high-dimensional-kaggle"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev,cde]'

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Tests and smoke tests

Run these three cells before training. Failures raise a visible cell error. The regressions
cover train-only normalization, temporal leakage, gap segmentation, model construction,
and checkpoint continuation. The data smoke report prints assets, split counts, dates, and
input shapes; the model smoke checks one training batch's forward pass, loss, and finite
gradients without an optimizer update. Test-window counts are metadata, not test evaluation.

In [ ]:
!pytest -q \
    tests/semantic/test_data_leakage.py \
    tests/semantic/test_window_continuity.py \
    tests/semantic/test_config_wiring.py \
    tests/unit/test_french_pipeline.py \
    tests/unit/test_french_portfolios.py \
    tests/unit/test_conditional_diffusion_model.py \
    tests/unit/test_cde.py \
    tests/unit/test_diffusion_training.py
assert _exit_code == 0, "Regression tests failed."

In [ ]:
!python scripts/data_smoke_test.py \
    --config configs/cde_ff49.yaml
assert _exit_code == 0, "Data smoke test failed."

In [ ]:
!python scripts/data_smoke_test.py \
    --config configs/cde_ff49.yaml \
    --model-smoke \
    --smoke-batch-size 1 \
    --device cuda \
    --report-json /kaggle/working/ff49_cde_smoke.json
assert _exit_code == 0, "Model smoke test failed."

In [ ]:
import json
import pandas as pd
from pathlib import Path

smoke = json.loads(Path("/kaggle/working/ff49_cde_smoke.json").read_text())
assert smoke["dataset"] == "ken_french_49" and smoke["n_assets"] == 49
display(pd.DataFrame([{
    "Dataset": smoke["dataset"], "Assets": smoke["n_assets"],
    "Train windows": smoke["train_samples"],
    "Validation windows": smoke["validation_samples"],
    "Test windows (count only)": smoke["test_samples"],
    "Validation history shape": smoke["history_shape"],
    "Prediction shape": smoke["model_smoke"]["prediction_shape"],
    "Smoke loss": smoke["model_smoke"]["loss"],
    "Finite gradients": smoke["model_smoke"]["gradients_finite"],
}]))

## Restore previous results

Restore only **`high_dimensional/ff49_cde_clean`** from `amirentezari/diffusion-portfolio`.
An existing local run is kept. If this experiment has no backup yet, training starts a new run.
Download errors remain visible; no unrelated experiment is copied into this run.
Use the saved configuration unchanged when resuming.
**Checkpoint safety:** The previous `/kaggle/working/ff49_cde` run has mismatched `best.pt` and `last.pt`. This notebook uses a new `ff49_cde_clean` directory and separate Kaggle backup namespace, leaving the old files untouched.


In [ ]:
from pathlib import Path
import shutil
import kagglehub
from diffusion_portfolio.config import load_config

HANDLE = "amirentezari/diffusion-portfolio"
EXPERIMENT = "high_dimensional/ff49_cde_clean"
RUN_DIR = Path("/kaggle/working/ff49_cde_clean")

artifact_path = Path(kagglehub.dataset_download(HANDLE))
source = artifact_path / EXPERIMENT
if RUN_DIR.exists():
    print("Already available:", RUN_DIR)
elif source.is_dir():
    shutil.copytree(source, RUN_DIR)
    print("Restored:", RUN_DIR)
else:
    print("No backup yet:", EXPERIMENT)

if RUN_DIR.exists():
    for filename in ("config.yaml", "standardizer.npz", "best.pt", "last.pt"):
        if not (RUN_DIR / filename).is_file():
            raise FileNotFoundError(RUN_DIR / filename)
    assert load_config(RUN_DIR / "config.yaml") == load_config("configs/cde_ff49.yaml"), \
        "Saved configuration differs; use a separate directory for a new experiment."

## Training

### Short training pilot (2 epochs)

Click this cell to run up to **absolute epoch 2**, using the unchanged 50-epoch configuration.
`--resume` creates `last.pt` even on a fresh run and restores model, optimizer, history, and RNG
state on continuation. `best.pt` tracks the lowest validation loss. The shared trainer prints epoch starts, progress every 10 batches (and the final batch),
and flushed epoch losses with elapsed time. Inspect the next plot before selecting full training.
Rerunning this pilot cell does not add two more epochs to a run already past epoch 2.

In [ ]:
!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/train_diffusion.py \
    --config configs/cde_ff49.yaml \
    --output-dir /kaggle/working/ff49_cde_clean \
    --device cuda \
    --resume \
    --stop-after-epoch 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# last.pt contains the most recent epoch history even if a run was interrupted.
import torch
if (RUN_DIR / "last.pt").exists():
    history = pd.DataFrame(
        torch.load(RUN_DIR / "last.pt", map_location="cpu", weights_only=False)["history"]
    )
else:
    history = pd.read_csv(RUN_DIR / "history.csv")
display(history)
best = history.loc[history["val_loss"].idxmin()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["epoch"], history["train_loss"], marker="o", label="Train Loss")
ax.plot(history["epoch"], history["val_loss"], marker="o", label="Validation Loss")
ax.scatter(best["epoch"], best["val_loss"], color="black", zorder=3,
           label=f"Best validation: epoch {int(best['epoch'])}")
ax.set(xlabel="Epoch", ylabel="Diffusion loss", title="FF49 — Neural CDE Diffusion")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print(f"Best validation epoch: {int(best['epoch'])}; loss: {best['val_loss']:.6f}")

### Full training (resume to 50 epochs)

Select this cell after reviewing the pilot. It resumes **the same directory** from `last.pt`
and continues to the configured epoch 50. A completed run with `summary.json` is skipped by
the CLI. After a Kaggle interruption, restore the backup and rerun this cell.

Rerun either loss-plot cell whenever needed. `history.csv` is refreshed when a training command
returns normally; after an interruption, resume training to refresh it from the history in
`last.pt`. `summary.json` is written only when full training completes.
Epoch and batch progress is printed directly by the shared trainer and flushed live. Do not mix the old damaged checkpoint directory with this run.

In [ ]:
!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/train_diffusion.py \
    --config configs/cde_ff49.yaml \
    --output-dir /kaggle/working/ff49_cde_clean \
    --device cuda \
    --resume

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# last.pt contains the most recent epoch history even if a run was interrupted.
import torch
if (RUN_DIR / "last.pt").exists():
    history = pd.DataFrame(
        torch.load(RUN_DIR / "last.pt", map_location="cpu", weights_only=False)["history"]
    )
else:
    history = pd.read_csv(RUN_DIR / "history.csv")
display(history)
best = history.loc[history["val_loss"].idxmin()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["epoch"], history["train_loss"], marker="o", label="Train Loss")
ax.plot(history["epoch"], history["val_loss"], marker="o", label="Validation Loss")
ax.scatter(best["epoch"], best["val_loss"], color="black", zorder=3,
           label=f"Best validation: epoch {int(best['epoch'])}")
ax.set(xlabel="Epoch", ylabel="Diffusion loss", title="FF49 — Neural CDE Diffusion")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print(f"Best validation epoch: {int(best['epoch'])}; loss: {best['val_loss']:.6f}")

import json

summary_path = RUN_DIR / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(pd.Series(summary, name="Training summary").to_frame())
else:
    print("Partial run: no final summary yet. Resume training or back up the checkpoint below.")

## Validation evaluation

Run after full training. The existing evaluator loads **`best.pt`**, the saved configuration,
and the saved training scaler. It uses the configured **256 scenarios**, validation seed
**12345**, and batch size **32**. Metrics use raw arithmetic excess returns. No held-out test
evaluation is performed. The completion check prevents presenting the pilot as a final run.

In [ ]:
assert (RUN_DIR / "summary.json").is_file(), "Complete full training before final validation evaluation."

!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --run-dir /kaggle/working/ff49_cde_clean \
    --split validation \
    --batch-size 32 \
    --device cuda

In [ ]:
import json
import pandas as pd

summary = json.loads((RUN_DIR / "summary.json").read_text())
metrics = json.loads((RUN_DIR / "validation_probabilistic_metrics.json").read_text())
assert metrics["split"] == "validation"
assert metrics["checkpoint_epoch"] == summary["best_epoch"]
calibration = pd.DataFrame(metrics["calibration"])
display(pd.DataFrame([{
    "Dataset": "ken_french_49", "Model": "Neural CDE", "Seed": 42,
    "Best epoch": metrics["checkpoint_epoch"],
    "Validation diffusion loss": metrics["checkpoint_val_loss"],
    "CRPS": metrics["crps_mean"], "Energy Score": metrics["energy_score"],
    "Mean |ACE|": calibration["ace"].abs().mean(),
    "Parameters": summary["parameter_count"],
    "Train windows": summary["train_samples"],
    "Validation windows": metrics["n_validation"],
    "Validation start": metrics["validation_start"],
    "Validation end": metrics["validation_end"],
    "Git commit": summary["git_commit"],
}]))
display(calibration.rename(columns={"level": "Nominal coverage", "picp": "PICP", "ace": "ACE"}))

## Optional FF49 comparison

This cell reads completed local FF49 Vanilla and Neural CDE runs if both are available;
it is not a training prerequisite. To compare in a fresh session, restore
`high_dimensional/ff49_vanilla` from the artifact dataset to
`/kaggle/working/ff49_vanilla` using the same restore pattern above.
The configurations match dataset, splits, seed, return convention, and evaluation protocol.
Lower CRPS, Energy Score, and mean absolute ACE are preferable; interpret actual saved results.

In [ ]:
runs = {
    "Vanilla / MLP": Path("/kaggle/working/ff49_vanilla"),
    "Neural CDE": RUN_DIR,
}
if all((run / name).is_file() for run in runs.values()
       for name in ("summary.json", "validation_probabilistic_metrics.json")):
    rows = []
    for name, run in runs.items():
        training = json.loads((run / "summary.json").read_text())
        evaluation = json.loads((run / "validation_probabilistic_metrics.json").read_text())
        rows.append({
            "Model": name, "Best epoch": training["best_epoch"],
            "Validation loss": training["best_val_loss"],
            "CRPS": evaluation["crps_mean"], "Energy Score": evaluation["energy_score"],
            "Mean |ACE|": pd.DataFrame(evaluation["calibration"])["ace"].abs().mean(),
            "Parameters": training["parameter_count"],
        })
    display(pd.DataFrame(rows).set_index("Model"))
else:
    print("Comparison available once both completed FF49 runs and validation metrics are local.")

## Backup

**Select the following cell when ready to upload**, including after the two-epoch pilot or
an interrupted session with a saved checkpoint. It requires `config.yaml`, `standardizer.npz`,
`best.pt`, and `last.pt`; a final summary is not required for a partial run.

The latest central dataset is downloaded first and copied in full, then only
**`high_dimensional/ff49_cde_clean`** is replaced. KF12 and all other experiment directories remain
in the upload. If downloading fails or returns an empty dataset, the cell stops before upload.
Run central-dataset updates one at a time to avoid concurrent uploads replacing each other.
Kaggle credentials must permit updating `amirentezari/diffusion-portfolio`.

In [ ]:
from pathlib import Path
import shutil
import kagglehub

for filename in ("config.yaml", "standardizer.npz", "best.pt", "last.pt"):
    if not (RUN_DIR / filename).is_file():
        raise FileNotFoundError(RUN_DIR / filename)

artifact_path = Path(kagglehub.dataset_download(HANDLE, force_download=True))
if not any(p.is_file() and p.name != ".complete" for p in artifact_path.rglob("*")):
    raise RuntimeError("Existing artifact dataset is empty; refusing to replace it.")

staging = RUN_DIR.parent / "artifact_sync"
if staging.exists():
    shutil.rmtree(staging)
shutil.copytree(artifact_path, staging)

destination = staging / EXPERIMENT
if destination.exists():
    shutil.rmtree(destination)
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_DIR, destination)

kagglehub.dataset_upload(
    HANDLE,
    str(staging),
    version_notes=f"Checkpoint and available results: {EXPERIMENT}",
    ignore_patterns=[".complete"],
)
print("Backed up:", EXPERIMENT)